# Modelo realista · todo lo posible sobre `real_todo` de Pablo (Adri · `realista/`)

**Qué es:** el LightGBM realista de Pablo (`realista/m_lgb_real.py`): ve el **snapshot actual y snapshots PASADOS** de la
sesión, como funcionaría en tiempo real (nada del futuro, nada de etiquetas de otros controladores). Su mejor versión,
`real_todo` (bloques `base + logic + past + causal + geo`, 3 semillas), da **CV 0.6703** y **0.6718** con su segundo paso causal.

**Qué se hace aquí:** una **búsqueda automática** que prueba, una a una y sobre lo que ya ha ganado, todas las mejoras con
opciones reales de sumar, y genera la versión final. Cada candidata sale de algo que ya funcionó en el equipo:

| Candidata | Qué cambia | De dónde sale |
|---|---|---|
| `peso_tramo` | al entrenar, cada snapshot pesa 1/√(longitud de su tramo) | `notebooks/LGB_adri.ipynb` 12c–12d: **+0.8 robusto** en el modelo de un snapshot |
| `peso_racha` | igual, con la longitud de la racha (no mira la etiqueta) | `LGB_adri` 13a (empató con el tramo) |
| `sin3` | quitar `a_n300`, `a_n60`, `seq_len` (la carga de la sesión, delata al controlador) | Juanma, `pruebas/notas/h4_sel.md` |
| `sel_juanma` | quitar las 23 columnas de Juanma (las 3 anteriores + 20 casi inútiles) | Juanma `h4`: +0.4, peor controlador +1.4 |
| `atencion` | **nueva**: cuántas instrucciones ha dado el controlador a ese avión en los últimos 60 / 180 s y en toda la sesión, vistas en snapshots pasados | idea de Pablo (el futuro gana por la *siguiente* instrucción): aquí, su espejo hacia atrás |
| `modo` | **nueva**: qué tipo fue la última instrucción del controlador (a cualquier avión) y hace cuánto | análisis de fallos: falla cuando el controlador está "planificando" en vez de "reaccionando" |
| `alerta_contexto` | las features `alerta` + `contexto` de `LGB_adri` | `LGB_adri` bloque 2 |
| `sin_logic_past` | quitar los bloques `logic` y `past` | Pablo: con pasado "no aportan"; menos ruido |
| `lento` | `lr=0.03`, 700 árboles | Pablo, futuro bloque 2 (mejoraba al peor controlador) |
| `colsample05` | `colsample_bytree=0.5` | Pablo, futuro 9b |
| `l63_et` | 63 hojas + `min_child=100` + `extra_trees` | Juanma, futuro 13a (+0.3 con futuro) |

**Criterio para quedarse una candidata** (el de Pablo y Juanma): sube la CV al menos `MIN_GAIN` **y** suben más sesiones de
las que bajan, frente a lo mejor hasta ese momento, con las **mismas 2 semillas**. Las candidatas del mismo grupo (`peso`,
`drop`, `feats`, `params`) son alternativas: se prueban todas y se queda la mejor que pase.

**Después**, la versión final con **10 semillas** y el **segundo paso causal** de Pablo → `real_adri` y `real_adri_ses` en
`realista/oof|preds|submissions`.

**Reglas:** mismos folds (`Data/folds.csv`) y código de Pablo (importado, sin modificarlo). Ninguna feature nueva mira al futuro
ni usa etiquetas; el peso usa la etiqueta solo de las filas de **entrenamiento**.

> ## Punto de partida actual (21/10, 03:00)
>
> **Mejor modelo realista: `real_todo` (Pablo) = CV 0.6703, 0.6718 con segundo paso causal**, peor controlador 0.600.
>
> | Paso | CV | peor controlador | estado | dónde |
> |---|---|---|---|---|
> | `real_todo` (Pablo, 3 semillas) | 0.6703 | 0.596 | referencia | `LGB_realista.ipynb` |
> | búsqueda automática (11 candidatas) | | | **siguiente** | 1 |
> | versión final `real_adri`, 10 semillas + segundo paso causal | | | **siguiente** | 2 |

## Preparación

**Cómo usarlo:** *Run All*. Con `RAPIDO = True` (bloque 1, por defecto) tarda **~20–25 min** en total: 1 semilla en la
búsqueda, solo las 5 candidatas más prometedoras y versión final con 5 semillas. Con `RAPIDO = False`, la búsqueda completa
(11 candidatas, 2 semillas, final con 10): ~60–75 min. Todo queda en `experiments/lgb_real_runs.csv` (prefijo `adri_`)
aunque se corte a medias. **Ctrl + S al terminar.**

In [ ]:
import json, os, sys, time, types
from pathlib import Path
import numpy as np, pandas as pd

# Los scripts de Pablo importan fcntl (bloqueo de archivos de Linux), que no existe en Windows: sustituto que no hace nada.
sys.modules.setdefault("fcntl", types.SimpleNamespace(flock=lambda *a, **k: None, LOCK_EX=2))

ROOT = Path.cwd().parent if Path.cwd().name in ("realista", "notebooks") else Path.cwd()
REAL = ROOT / "realista"
for p in ("cv", "futuro", "realista"):
    sys.path.insert(0, str(ROOT / p))
import m_lgb_real as R, m_lgb_future as M, feat_session_causal, metricas
from metric import save_outputs

t0 = time.time()
data = R.build_data()                       # features de Pablo (~3-5 min)
lab, te, fs = data["lab"], data["te"], data["fs"]
M.add_lengths(lab)                          # n_tramo, n_racha: solo para pesar el entrenamiento
_, P = R.parse(["--threads", str(os.cpu_count() or 8)])   # parámetros de Pablo (los de EQUIPO.md)
FEATS = "base,logic,past,causal,geo"       # los bloques de real_todo
print("bloques de Pablo:", {k: len(v) for k, v in fs.items()}, f"({time.time() - t0:.0f}s)")

### Features nuevas

| Grupo | Columnas | Qué mide | Solo mira atrás porque… |
|---|---|---|---|
| `atencion` | `at_n60`, `at_n180`, `at_nses`, `at_n180_rk` | instrucciones del controlador a ese avión en los últimos 60 / 180 s y en la sesión; ranking en el snapshot | cada instrucción se cuenta desde el primer snapshot en que se ve (`t − a_since_flight`), y solo si ese snapshot es el actual o anterior |
| `modo` | `md_last_*`, `md_since` | tipo de la última instrucción del controlador a **cualquier** avión del snapshot y hace cuánto | usa solo columnas del snapshot actual |
| `alerta` | `al_*` | conflictos con alerta STCA, nº de alertas, ranking del tiempo al mínimo entre conflictos | snapshot actual |
| `contexto` | `cx_*` | nº de aviones y de transfers / assumes / zonas / conflictos pendientes en el snapshot | snapshot actual |

In [ ]:
ACTS = ["a_last_assume_control", "a_last_force_assume_control", "a_last_assign_level_clearance",
        "a_last_assign_lateral_clearance", "a_last_initiate_transfer_of_control"]
CONFLICT_TYPES = [0, 1, 6]


def fe_atencion(d):
    k = ["session", "cs"]
    x = d.drop_duplicates(k + ["t"])[k + ["t", "a_since_flight"]].reset_index(drop=True)
    x["ev"] = np.where(x.a_since_flight < 3600, np.round(x.t - x.a_since_flight), np.nan)
    ev = x.dropna(subset=["ev"]).groupby(k + ["ev"]).t.min().rename("t_obs").reset_index()  # primera vez que se ve
    m = x[k + ["t"]].reset_index().merge(ev, on=k)
    vista = m.t_obs <= m.t                                          # solo instrucciones ya vistas
    m["n60"], m["n180"], m["nses"] = vista & (m.ev > m.t - 60), vista & (m.ev > m.t - 180), vista
    c = m.groupby("index")[["n60", "n180", "nses"]].sum()
    x[["at_n60", "at_n180", "at_nses"]] = c.reindex(x.index).fillna(0).values
    out = d[k + ["t"]].merge(x[k + ["t", "at_n60", "at_n180", "at_nses"]], on=k + ["t"], how="left")
    out.index = d.index
    out["at_n180_rk"] = out.at_n180.groupby(d.sample_id).rank(ascending=False, method="min")
    return out[["at_n60", "at_n180", "at_nses", "at_n180_rk"]].astype("float32")


def fe_modo(d):
    i = d.groupby("sample_id").a_since_flight.idxmin()               # avión con la instrucción más reciente
    last = d.loc[i, ["sample_id", "a_since_flight"] + ACTS].set_index("sample_id")
    last.columns = ["md_since"] + ["md_last_" + c[7:] for c in ACTS]
    return last.reindex(d.sample_id).set_index(d.index).astype("float32")


def fe_alerta(d):
    conf = d.task_index.isin(CONFLICT_TYPES)
    stca = d.HMI_conflict_safety_alert == 1
    return pd.DataFrame({
        "al_conflict_stca": (conf & stca).astype(int),
        "al_n_stca": stca.astype(int).groupby(d.sample_id).transform("sum"),
        "al_ttmd_rank": d.time_to_MD.where(conf).groupby(d.sample_id).rank(method="min"),
        "al_conflict_multi": (conf & (d.multiple_conflicts_ind == 1)).astype(int),
        "al_conflict_dmd": d.distance_to_MD.where(conf)}, index=d.index)


def fe_contexto(d):
    t = d.task_index
    tmp = pd.DataFrame({"assume_offer": (t == 4) & (d.x_transfer_in == 1),
                        "transfer_due": (t == 5) & (d.k_aircraftNeedsToBeTransferred == 1),
                        "zone": t == 12, "conflict": t.isin(CONFLICT_TYPES), "entry_coord": t == 2}, index=d.index)
    out = {"cx_n_aircraft": d.groupby("sample_id").cs.transform("nunique")}
    for c in tmp.columns:
        out["cx_n_" + c] = tmp[c].astype(int).groupby(d.sample_id).transform("sum")
    return pd.DataFrame(out, index=d.index)


t0 = time.time()
for nombre, fn in [("atencion", fe_atencion), ("modo", fe_modo), ("alerta", fe_alerta), ("contexto", fe_contexto)]:
    nuevas_lab, nuevas_te = fn(lab), fn(te)
    for c in nuevas_lab.columns:
        lab[c], te[c] = nuevas_lab[c].values, nuevas_te[c].values
    fs[nombre] = list(nuevas_lab.columns)
print({k: len(fs[k]) for k in ("atencion", "modo", "alerta", "contexto")}, f"({time.time() - t0:.0f}s)")

### Entrenamiento y comparación
`cv_real` entrena con los mismos folds, parámetros y semillas que Pablo, con lo que diga la configuración (bloques, columnas
quitadas, peso, parámetros), y lo compara con `metricas.py` de Pablo frente a una referencia. `guardar=True` entrena también
con todo y escribe `realista/oof|preds|submissions`; `sesion=True` añade el segundo paso causal de Pablo.

In [ ]:
OOF, RES = {}, {}                           # nombre -> oof; nombre -> métricas


def config(feats=FEATS, extra=(), drop=(), peso=None, params=None):
    return dict(feats=feats, extra=list(extra), drop=list(drop), peso=peso, params=dict(params or {}))


def cv_real(nombre, cfg, seeds=2, ref=None, guardar=False, sesion=False, tag=""):
    t0 = time.time()
    bloques = cfg["feats"].split(",") + cfg["extra"]
    F = [c for b in bloques for c in fs[b] if c not in set(cfg["drop"])]
    PP = types.SimpleNamespace(**{**vars(P), **cfg["params"]})
    w = None if cfg["peso"] is None else 1.0 / lab["n_" + cfg["peso"]].values ** 0.5
    grp = lambda A: A.groupby("sample_id", sort=False).size().values
    fit = lambda A, wa, s: M.model(PP, s).fit(A[F], A.target_index, group=grp(A), sample_weight=wa,
                                             categorical_feature=["task_index"])
    oof = np.zeros(len(lab))
    for f in sorted(lab.fold.unique()):
        tr, va = (lab.fold != f).values, (lab.fold == f).values
        for s in range(seeds):
            oof[va] += fit(lab[tr], None if w is None else w[tr], s).predict(lab.loc[va, F]) / seeds
    oof_df = lab[["sample_id", "row_index"]].assign(score=oof)
    OOF[nombre] = oof_df
    RES[nombre] = met = metricas.metricas(oof_df, OOF.get(ref))
    print(metricas.resumen(nombre, met, ref), f"  [{len(F)} features, {time.time() - t0:.0f}s]", flush=True)
    a = types.SimpleNamespace(tag=tag or json.dumps(cfg), feats=",".join(bloques), drop=",".join(cfg["drop"]),
                              seeds=seeds, sesion=sesion)
    R.log_run(a, nombre, met, len(F), time.time() - t0)
    if guardar:
        pred = np.mean([fit(lab, w, s).predict(te[F]) for s in range(seeds)], axis=0)
        save_outputs(nombre, oof_df, te[["sample_id", "row_index"]].assign(score=pred), out=REAL)
        if sesion:
            so, sp, pesos = feat_session_causal.sesion(lab, oof, te, pred)
            o2 = lab[["sample_id", "row_index"]].assign(score=so)
            OOF[nombre + "_ses"] = o2
            RES[nombre + "_ses"] = m2 = metricas.metricas(o2, OOF.get(ref + "_ses") if ref else None)
            print(metricas.resumen(nombre + "_ses", m2, ref + "_ses" if ref else None), flush=True)
            print("  pesos del segundo paso causal:", pesos)
            save_outputs(nombre + "_ses", o2, te[["sample_id", "row_index"]].assign(score=sp), out=REAL)
            R.log_run(a, nombre + "_ses", m2, len(F), time.time() - t0)
    return met


def tabla(nombres, ref):
    filas = []
    for n in nombres:
        m = metricas.metricas(OOF[n], OOF[ref] if n != ref else None)
        filas.append(dict(modelo=n, cv=m["cv"], peor_atco=m["min_atco"], mediana_sesion=m["ses_median"],
                          esc1=m.get("sc1"), esc2=m.get("sc2"), esc3=m.get("sc3"), d_cv=m.get("d_cv"),
                          sesiones_suben=m.get("ses_up"), sesiones_bajan=m.get("ses_down"),
                          controladores_suben=m.get("atco_up"), controladores_bajan=m.get("atco_down")))
    return pd.DataFrame(filas).round(4)

## 1 · Búsqueda automática

Empieza por `real_todo` (`adri_base`) con las mismas semillas que las candidatas. Recorre los grupos en orden; dentro de cada
grupo prueba todas las alternativas **sobre lo mejor hasta ese momento** y se queda con la que más suba, si cumple el criterio.
`RAPIDO = True`: 6 entrenamientos de ~2 min (~12 min). `RAPIDO = False`: 12 de ~3–4 min (~40 min).

In [ ]:
RAPIDO = True        # True: ~20-25 min en total. False: búsqueda completa, ~60-75 min
SEEDS_BUSQ = 1 if RAPIDO else 2
MIN_GAIN = 0.003 if RAPIDO else 0.002     # con 1 semilla el ruido es mayor: se pide más
QUITAR_JUANMA = json.load(open(ROOT / "pruebas" / "notas" / "h4_quitar.json"))
GRUPOS = [   # (grupo, [(nombre, cambios), ...]): las alternativas de un grupo se excluyen entre sí
    ("peso",   [("peso_tramo", dict(peso="tramo")), ("peso_racha", dict(peso="racha"))]),
    ("drop",   [("sin3", dict(drop=["a_n300", "a_n60", "seq_len"])), ("sel_juanma", dict(drop=QUITAR_JUANMA))]),
    ("atencion", [("atencion", dict(extra=["atencion"]))]),
    ("modo",   [("modo", dict(extra=["modo"]))]),
    ("alerta_contexto", [("alerta_contexto", dict(extra=["alerta", "contexto"]))]),
    ("feats",  [("sin_logic_past", dict(feats="base,causal,geo"))]),
    ("params", [("lento", dict(params=dict(lr=0.03, n_estimators=700))),
                ("colsample05", dict(params=dict(colsample=0.5))),
                ("l63_et", dict(params=dict(leaves=63, min_child=100, extra_trees=True)))]),
]
if RAPIDO:   # las 5 más prometedoras: peso por tramo, quitar la carga de la sesión, las 2 nuevas y colsample
    QUEDAN = {"peso_tramo", "sin3", "atencion", "modo", "colsample05"}
    GRUPOS = [(g, [x for x in alts if x[0] in QUEDAN]) for g, alts in GRUPOS]
    GRUPOS = [(g, alts) for g, alts in GRUPOS if alts]


def aplicar(cfg, cambios):
    c = json.loads(json.dumps(cfg))
    for k, v in cambios.items():
        if k == "extra":
            c["extra"] = c["extra"] + list(v)
        elif k == "params":
            c["params"] = {**c["params"], **v}
        else:
            c[k] = v
    return c


mejor_cfg, mejor = config(), "adri_base"
cv_real(mejor, mejor_cfg, seeds=SEEDS_BUSQ, tag=f"real_todo con {SEEDS_BUSQ} semilla(s) (referencia)")
historia = [dict(paso="base", candidata="adri_base", cv=RES[mejor]["cv"], d_cv=0.0, sesiones="", veredicto="referencia")]
for grupo, alternativas in GRUPOS:
    ganadora = None
    for nombre, cambios in alternativas:
        cfg = aplicar(mejor_cfg, cambios)
        n = f"adri_{nombre}"
        m = cv_real(n, cfg, seeds=SEEDS_BUSQ, ref=mejor, tag=f"búsqueda: {nombre} sobre {mejor}")
        pasa = m["d_cv"] >= MIN_GAIN and m["ses_up"] > m["ses_down"]
        historia.append(dict(paso=grupo, candidata=n, cv=m["cv"], d_cv=m["d_cv"],
                             sesiones=f"{m['ses_up']}/{m['ses_down']}", veredicto="pasa" if pasa else "no"))
        if pasa and (ganadora is None or m["cv"] > RES[ganadora[0]]["cv"]):
            ganadora = (n, cfg)
    if ganadora:
        mejor, mejor_cfg = ganadora
        print(f"  >>> se queda {mejor}  (CV {RES[mejor]['cv']:.4f})", flush=True)
        next(h for h in historia if h["candidata"] == mejor)["veredicto"] = "SE QUEDA"
    else:
        print(f"  >>> {grupo}: nada pasa el criterio", flush=True)

print("\nCONFIGURACIÓN GANADORA:", json.dumps(mejor_cfg))
json.dump(dict(nombre=mejor, cfg=mejor_cfg, cv=RES[mejor]["cv"]), open(REAL / "adri_busqueda.json", "w"), indent=1)
pd.DataFrame(historia).round(4)

In [ ]:
tabla(["adri_base", mejor], "adri_base")

**Conclusión bloque 1 · (rellenar tras ejecutar)**

| Paso | Candidata | CV | Δ vs lo mejor hasta entonces | sesiones suben / bajan | |
|---|---|---|---|---|---|
| … | | | | | |

## 2 · Versión final: 5–10 semillas + segundo paso causal

La configuración ganadora del bloque 1 (si se ha reiniciado el kernel, se lee de `realista/adri_busqueda.json`) con **5
semillas** (`RAPIDO`) o **10**, y el **segundo paso causal** de Pablo. Guarda `real_adri` y `real_adri_ses` en `realista/oof|preds|submissions`.
Con `COMPARAR_BASE10 = True` genera también `real_todo` con las mismas semillas para comparar en igualdad. ~8–10 min con 5
semillas, ~20 min con 10.

In [ ]:
GENERAR_FINAL = True
COMPARAR_BASE10 = False
SEEDS_FINAL = 5 if globals().get("RAPIDO", True) else 10

if GENERAR_FINAL:
    if "mejor_cfg" not in globals():
        b = json.load(open(REAL / "adri_busqueda.json")); mejor_cfg = b["cfg"]
    ref10 = None
    if COMPARAR_BASE10:
        cv_real("adri_base10", config(), seeds=SEEDS_FINAL, tag=f"real_todo con {SEEDS_FINAL} semillas")
        ref10 = "adri_base10"
    cv_real("real_adri", mejor_cfg, seeds=SEEDS_FINAL, ref=ref10, guardar=True, sesion=True, tag="versión final de Adri")
    s = pd.read_csv(REAL / "submissions" / "sub_real_adri_ses.csv")
    print("submission real_adri_ses:", len(s), "filas | sample_id únicos:", s.sample_id.is_unique,
          "| 3 ids por fila:", s.prediction.str.split().str.len().eq(3).all())
else:
    print("No se genera (GENERAR_FINAL = False).")

**Resultado bloque 2 · (rellenar tras ejecutar)**

| | CV | peor controlador | mediana por sesión | Kaggle público |
|---|---|---|---|---|
| `real_todo` (Pablo, 3 semillas) | 0.6703 | 0.596 | | |
| `real_adri` (10 semillas) | | | | |
| `real_adri_ses` | | | | |

Submission para Kaggle: `realista/submissions/sub_real_adri_ses.csv` (o `sub_real_adri.csv` si el segundo paso no ayuda).